# Algoritmo de enfriamiento simulado aplico con los mejores parámetros encontrados en test

In [ ]:
import random
random.seed(42)
n_objetos = 200
capacidad_mochila = 600

limite_objetos = n_objetos // 3

objetos_peso = [random.randint(3, 60) for _ in range(n_objetos)]
objetos_valor = [random.randint(10, 150) for _ in range(n_objetos)]
objetos_cantidad = [random.randint(2, 15) for _ in range(n_objetos)]

print("Pesos:", objetos_peso)
print("Valores:", objetos_valor)
print("Cantidades:", objetos_cantidad)
print("Capacidad mochila:", capacidad_mochila)
capacidad_maxima_objetos = [capacidad_mochila // w if w > 0 else capacidad_mochila for w in objetos_peso]

N = len(objetos_peso)

Pesos: [43, 10, 4, 50, 20, 18, 17, 11, 50, 9, 46, 50, 60, 37, 8, 40, 30, 5, 4, 8, 16, 17, 35, 41, 4, 38, 15, 48, 44, 47, 37, 29, 17, 31, 40, 20, 54, 58, 3, 51, 54, 13, 47, 30, 24, 20, 12, 16, 51, 24, 9, 8, 27, 9, 25, 57, 25, 41, 19, 54, 5, 49, 32, 37, 10, 27, 8, 38, 21, 56, 43, 42, 59, 58, 26, 39, 15, 48, 7, 5, 45, 17, 52, 21, 8, 57, 17, 58, 9, 27, 20, 32, 43, 56, 26, 13, 26, 25, 16, 45, 20, 47, 46, 44, 7, 41, 43, 13, 37, 49, 18, 13, 32, 27, 20, 43, 47, 38, 17, 46, 23, 56, 52, 52, 6, 17, 55, 5, 54, 23, 28, 20, 7, 16, 39, 59, 48, 23, 16, 44, 34, 28, 59, 44, 32, 12, 19, 11, 18, 50, 38, 37, 19, 50, 40, 30, 60, 40, 28, 26, 17, 11, 35, 34, 8, 51, 6, 58, 10, 12, 43, 13, 53, 46, 30, 41, 7, 27, 27, 41, 32, 36, 19, 38, 58, 3, 46, 49, 10, 46, 59, 37, 51, 20, 52, 44, 24, 10, 21, 30]
Valores: [50, 126, 10, 77, 138, 55, 139, 37, 86, 139, 60, 49, 105, 51, 148, 145, 10, 92, 135, 14, 38, 102, 88, 71, 24, 71, 30, 31, 134, 27, 146, 42, 42, 131, 150, 52, 77, 145, 118, 64, 148, 61, 89, 112, 105, 122, 142,

In [ ]:
def create_restricciones():
    restricciones = {"separados": {}, "juntos": {}}

    for _ in range(int(n_objetos // 2)):
        # Primero generamos las restricciones de que un objeto no puede estar con otro
        objeto = random.randint(0, N - 1)
        otro_objeto = random.randint(0, N - 1)
        while otro_objeto == objeto:
            otro_objeto = random.randint(0, N - 1)
        if objeto not in restricciones["separados"]:
            restricciones["separados"][objeto] = set()
        restricciones["separados"][objeto].add(otro_objeto)

    for _ in range(int(n_objetos // 2)):
        objeto = random.randint(0, N - 1)
        otro_objeto = random.randint(0, N - 1)
        while otro_objeto == objeto:
            otro_objeto = random.randint(0, N - 1)
        if objeto not in restricciones["juntos"]:
            restricciones["juntos"][objeto] = set()
        restricciones["juntos"][objeto].add(otro_objeto)

    return restricciones

restricciones = create_restricciones()
print("Restricciones:", restricciones)

Restricciones: {'separados': {121: {156, 188}, 43: {168}, 21: {72}, 131: {169}, 162: {158}, 85: {23}, 192: {107, 60, 63}, 172: {79}, 57: {50, 166}, 37: {6}, 11: {62}, 196: {18, 199}, 116: {106, 83}, 161: {147}, 49: {183}, 178: {98, 132, 5}, 126: {102}, 62: {37}, 167: {176}, 1: {192, 90}, 197: {27, 149}, 199: {108}, 56: {45}, 118: {12}, 142: {63}, 31: {116}, 34: {118}, 170: {135}, 143: {152}, 81: {193, 138}, 113: {156}, 184: {129}, 109: {140}, 114: {40}, 190: {121}, 115: {66}, 163: {70}, 133: {124}, 160: {61}, 70: {112}, 19: {182}, 73: {60}, 69: {85, 111}, 20: {35}, 38: {59}, 98: {177}, 39: {180, 61}, 54: {16}, 106: {104}, 84: {138}, 119: {106}, 15: {52}, 107: {99}, 195: {147}, 97: {122, 71}, 76: {192}, 99: {107, 86}, 137: {191}, 188: {139}, 154: {56}, 124: {56, 7}, 171: {173}, 103: {185}, 42: {119}, 32: {159}, 136: {6}, 100: {151}, 144: {169}, 6: {21}, 164: {109}, 46: {12}, 66: {97}, 83: {54}, 86: {194}, 64: {20}, 120: {4}, 191: {138}, 13: {89}, 17: {199}, 166: {10}, 193: {7}, 63: {51}

In [ ]:
def validar(individuo):

    restricciones_violadas = []
    # Validar restricciones de que ciertos objetos no pueden estar juntos
    for objeto, separados in restricciones["separados"].items():
        if individuo[objeto] > 0:
            for otro_objeto in separados:
                if individuo[otro_objeto] > 0:
                    restricciones_violadas.append("separados")
                    break

    # Validar restricciones de que ciertos objetos deben estar juntos
    for objeto, juntos in restricciones["juntos"].items():
        if individuo[objeto] > 0:
            for otro_objeto in juntos:
                if individuo[otro_objeto] == 0:
                    #print("Falla en juntos:", objeto, otro_objeto)
                    restricciones_violadas.append("juntos")
                    break

    if len(restricciones_violadas) == 0:
        return True, []
    else:
        return False, restricciones_violadas

In [ ]:
import random
def populate():
    individual = []
    remaining_w = capacidad_mochila
    remaining_c = limite_objetos
    for i in range(N):
        maxx = min(objetos_cantidad[i], remaining_w // objetos_peso[i], remaining_c)
        cant = random.randint(0, maxx)
        individual.append(cant)
        remaining_w -= cant * objetos_peso[i]
        remaining_c -= cant
    return individual

In [ ]:
def fitness(individuo):
    peso_total = 0
    valor_total = 0
    cantidad_total = 0

    for i in range(N):
        peso_total += individuo[i] * objetos_peso[i]
        valor_total += individuo[i] * objetos_valor[i]
        cantidad_total += individuo[i]

    valido, tipo_violacion = validar(individuo)

    penalizacion = 0

    # Penalización por exceso de peso
    if peso_total > capacidad_mochila:
        exceso_peso = peso_total - capacidad_mochila
        penalizacion += exceso_peso * 10

    # Penalización por exceso de objetos
    if cantidad_total > limite_objetos:
        exceso_objetos = cantidad_total - limite_objetos
        penalizacion += exceso_objetos * 50

    # Penalizaciones por violación de restricciones
    if not valido:
        penalizaciones_por_tipo = {
            "separados": 1000,
            "juntos": 1000,
        }
        for v in tipo_violacion:
            penalizacion += penalizaciones_por_tipo.get(v, 1000)

    return valor_total - penalizacion

In [ ]:
import copy
def reparar(individuo):
    """
    Función para reparar un individuo que no cumple con las restricciones
    Se repara el individuo primero de todo arreglando los límites de cantidad máxima de cada objeto
    Luego se arreglan las restricciones de separados y juntos
    Luego se arregla el peso total y la cantidad total de objetos
    """
    individuo_reparado = individuo[:]

    # 1. Limitar cantidades máximas
    for i in range(len(individuo_reparado)):
        individuo_reparado[i] = min(individuo_reparado[i], objetos_cantidad[i])

    # 2. Reparar restricciones de separación
    individuo_reparado = reparar_separados(individuo_reparado)

    # 3. Reparar restricciones de juntos
    individuo_reparado = reparar_juntos(individuo_reparado)

    # 4. Ajustar peso
    individuo_reparado = ajustar_peso(individuo_reparado)

    # 5. Ajustar cantidad de objetos
    individuo_reparado = ajustar_cantidad_objetos(individuo_reparado)

    return individuo_reparado

def reparar_separados(individuo):
    """
    Repara violaciones de restricciones de separación
    Si hay objetos que estan juntos y no deberían, elimino el que tiene peor ratio valor/peso
    Se eliminan por peor ratio valor/peso porque así maximizamos el valor del individuo
    """
    individuo_reparado = individuo[:]

    for objeto, separados in restricciones["separados"].items():
        if individuo_reparado[objeto] > 0:
            for otro_objeto in separados:
                if individuo_reparado[otro_objeto] > 0:
                    # Elegir cuál eliminar basado en ratio valor/peso
                    ratio1 = objetos_valor[objeto] / objetos_peso[objeto]
                    ratio2 = objetos_valor[otro_objeto] / objetos_peso[otro_objeto]

                    if ratio1 < ratio2:
                        individuo_reparado[objeto] = 0
                    else:
                        individuo_reparado[otro_objeto] = 0

    return individuo_reparado

def reparar_juntos(individuo):
    """
    Repara violaciones de restricciones de juntos
    Si hay objetos que deberían estar juntos y no lo están, los añado si es posible
    Si no es posible porque sobrepase el peso o el limite de objetos, elimino el objeto principal, así me aseguro de que se cumpla la restricción
    """
    individuo_reparado = individuo[:]
    peso_actual = sum(individuo_reparado[i] * objetos_peso[i] for i in range(N))
    cantidad_actual = sum(individuo_reparado)

    for objeto, juntos in restricciones["juntos"].items():
        if individuo_reparado[objeto] > 0:
            for otro_objeto in juntos:
                if individuo_reparado[otro_objeto] == 0:
                    # Intentar añadir el objeto requerido
                    peso_adicional = objetos_peso[otro_objeto]

                    if (peso_actual + peso_adicional <= capacidad_mochila and
                        cantidad_actual + 1 <= limite_objetos):
                        # Podemos añadir el objeto requerido
                        individuo_reparado[otro_objeto] = 1
                        peso_actual += peso_adicional
                        cantidad_actual += 1
                    else:
                        # No podemos añadir el objeto requerido, eliminar el objeto principal
                        peso_actual -= individuo_reparado[objeto] * objetos_peso[objeto]
                        cantidad_actual -= individuo_reparado[objeto]
                        individuo_reparado[objeto] = 0

    return individuo_reparado

def ajustar_peso(individuo):
    """
    Ajusta el peso total eliminando objetos con peor ratio valor/peso
    Si el peso total es mayor que la capacidad de la mochila tengo que eliminar objetos hasta que el peso total sea menor o igual a la capacidad
    Primero obtengo los objetos ordenador de peor a mejor ratio valor/peso y los voy intentado eliminar para así maximizar el valor del individuo
    Si el objeto está en alguna restricción de juntos, solo lo elimino si tiene más de una unidad

    """
    individuo_reparado = individuo[:]
    peso_total = sum(individuo_reparado[i] * objetos_peso[i] for i in range(len(individuo_reparado)))

    if peso_total <= capacidad_mochila:
        return individuo_reparado

    # Ordenar por ratio valor/peso (peor primero)
    ratios = [(i, objetos_valor[i] / objetos_peso[i])
              for i in range(len(individuo_reparado)) if individuo_reparado[i] > 0]
    ratios.sort(key=lambda x: x[1])

    for i, _ in ratios:
        while individuo_reparado[i] > 0 and peso_total > capacidad_mochila:
            # Verificar si eliminar este objeto viola restricciones de juntos
            puede_eliminar = True
            for obj_principal, objetos_juntos in restricciones["juntos"].items():
                if (i in objetos_juntos and individuo_reparado[obj_principal] > 0 and
                    individuo_reparado[i] == 1):
                    # Este es el último objeto requerido, no podemos eliminarlo
                    puede_eliminar = False
                    break

            if puede_eliminar:
                individuo_reparado[i] -= 1
                peso_total -= objetos_peso[i]
            else:
                break

        if peso_total <= capacidad_mochila:
            break

    return individuo_reparado

def ajustar_cantidad_objetos(individuo):
    """
    Ajusta la cantidad total de objetos
    Esta función es parecida a la de ajustar_peso, solo que en lugar de arreglar la restricción del peso areglamos la restricción de cantidad total de objetos
    """
    individuo_reparado = individuo[:]
    cantidad_total = sum(individuo_reparado)

    if cantidad_total <= limite_objetos:
        return individuo_reparado

    # Ordenar por ratio valor/peso (peor primero)
    ratios = [(i, objetos_valor[i] / objetos_peso[i])
              for i in range(len(individuo_reparado)) if individuo_reparado[i] > 0]
    ratios.sort(key=lambda x: x[1])

    for i, _ in ratios:
        while individuo_reparado[i] > 0 and cantidad_total > limite_objetos:
            # Verificar restricciones de juntos antes de eliminar
            puede_eliminar = True
            for obj_principal, objetos_juntos in restricciones["juntos"].items():
                if (i in objetos_juntos and individuo_reparado[obj_principal] > 0 and
                    individuo_reparado[i] == 1):
                    puede_eliminar = False
                    break

            if puede_eliminar:
                individuo_reparado[i] -= 1
                cantidad_total -= 1
            else:
                break

        if cantidad_total <= limite_objetos:
            break

    return individuo_reparado

In [ ]:
import random
from math import exp

def simulated_annealing_aleatorizado(individuo, objetos_cantidad, fitness, validar,
                                    T_inicial=50, T_min=1e-3, alpha=0.95,
                                    metodo='primero', max_iter=5000, verbose=False):
    fitness_history = []
    mejor = individuo[:]
    individuo_actual = individuo[:]
    fitness_actual = fitness(individuo_actual)
    max_fitness = fitness_actual

    generaciones_sin_mejora = 0
    max_generaciones_sin_mejora = 500
    T = T_inicial
    i = 0

    while i < max_iter and T > T_min:
        cambio = False

        # Generar len(individuo) vecinos
        indices = list(range(len(individuo_actual)))
        random.shuffle(indices)
        for j in indices:
            for k in [-1, 1]:
              prev = individuo_actual[j]

              # Generamos el vecino respetando límites
              if individuo_actual[j] == 0 and k == -1:
                  continue
              if k == 1 and individuo_actual[j] == objetos_cantidad[j]:
                  continue

              individuo_actual[j] += k
              nuevo_fitness = fitness(individuo_actual)
              diferencia = nuevo_fitness - fitness_actual

              # Criterio de aceptación
              if diferencia > 0 or random.random() < exp(diferencia / T):
                  fitness_actual = nuevo_fitness
                  if nuevo_fitness > max_fitness:
                      max_fitness = nuevo_fitness
                      mejor = individuo_actual[:]
                      cambio = True
              else:
                  individuo_actual[j] = prev

        # Control de estancamiento
        if not cambio:
            generaciones_sin_mejora += 1
        else:
            generaciones_sin_mejora = 0

        if generaciones_sin_mejora >= max_generaciones_sin_mejora:
            # Borrar individuo y crear uno nuevo
            individuo_actual = populate()
            fitness_actual = fitness(individuo_actual)
            generaciones_sin_mejora = 0
            if verbose:
                print(f"Estancado {max_generaciones_sin_mejora} iteraciones")

        if metodo == 'primero':
          T = T -i*alpha
        elif metodo == 'segundo':
          T = T * alpha
        else:
          T = T/(1 + alpha*T)
        i += 1
        fitness_history.append(fitness_actual)

        if verbose:
            print(f"Iteración {i} | T={T:.4f} | Mejor fitness={max_fitness} | Es válido: {validar(mejor)}")

    return mejor, max_fitness, fitness_history


In [ ]:
T_inicial = 100
alpha = 0.99
metodo = "segundo"
mejor_solucion_fit = float('-inf')
fitness_history_soluciones = []
mejor_solucion = None

for i in range(100):
    individuo = populate()
    mejor, max_fit, fitness_history = simulated_annealing_aleatorizado(individuo, objetos_cantidad, fitness, validar, T_inicial=T_inicial,metodo=metodo,alpha=alpha)

    if max_fit > mejor_solucion_fit:
      print(max_fit, mejor_solucion_fit)
      mejor_solucion_fit = max_fit
      mejor_solucion = mejor
      fitness_history_soluciones = fitness_history

    print(f"Mejor fitness: {max_fit} - Es válido: {validar(mejor)} - {i+1}/100")


8685 -inf
Mejor fitness: 8685 - Es válido: (True, []) - 1/100
8983 8685
Mejor fitness: 8983 - Es válido: (True, []) - 2/100
Mejor fitness: 8786 - Es válido: (True, []) - 3/100
Mejor fitness: 8907 - Es válido: (True, []) - 4/100
Mejor fitness: 8983 - Es válido: (True, []) - 5/100
Mejor fitness: 8676 - Es válido: (True, []) - 6/100
Mejor fitness: 8983 - Es válido: (True, []) - 7/100
9056 8983
Mejor fitness: 9056 - Es válido: (True, []) - 8/100
Mejor fitness: 8983 - Es válido: (True, []) - 9/100
Mejor fitness: 8401 - Es válido: (True, []) - 10/100
Mejor fitness: 8983 - Es válido: (True, []) - 11/100
Mejor fitness: 8983 - Es válido: (True, []) - 12/100
Mejor fitness: 8758 - Es válido: (True, []) - 13/100
Mejor fitness: 9034 - Es válido: (True, []) - 14/100
Mejor fitness: 9037 - Es válido: (True, []) - 15/100
Mejor fitness: 8907 - Es válido: (True, []) - 16/100
Mejor fitness: 9049 - Es válido: (True, []) - 17/100
Mejor fitness: 8983 - Es válido: (True, []) - 18/100
Mejor fitness: 8790 - Es 

In [ ]:
if mejor_solucion is None:
    print("No se encontró una solución válida.")
else:
    mejor_solucion = reparar(mejor_solucion)
    mejor_solucion_fit = fitness(mejor_solucion)
    print(mejor_solucion)
    print(mejor_solucion_fit)
    print(validar(mejor_solucion))
    peso_total = 0
    valor_total = 0
    cantidad_total = 0

    for i in range(len(mejor_solucion)):
        peso_total += mejor_solucion[i] * objetos_peso[i]
        valor_total += mejor_solucion[i] * objetos_valor[i]
        cantidad_total += mejor_solucion[i]
    print()
    print(f"Valor total:{valor_total}")
    print(f"Peso total:{peso_total} - Capacidad mochila:{capacidad_mochila}")
    print(f"Cantidad total:{cantidad_total} - Límite objetos:{limite_objetos}")

[0, 12, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 6, 0, 0, 0, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
8724
(True, [])

Valor total:8724
Peso total:554 - Capacidad mochila:600
Cantidad total:66 - Límite objetos:66
